# eda.ipynb
## Análisis Exploratorio de Datos (EDA)
### TFG — Análisis de biomarcadores acústicos en ELA

**Autor:** Jakub Wysocki  

---

Este notebook realiza el análisis exploratorio completo del dataset consolidado.
Los resultados de este EDA justifican las decisiones metodológicas del Capítulo 3
de la memoria (selección de features, estrategia de normalización, gestión del
desbalanceo de clases).

**Estructura:**
1. Carga y visión general del dataset
2. Análisis de la variable objetivo (distribución de clases)
3. Análisis demográfico (género × clase)
4. Calidad de datos (nulos, duplicados, varianza cero)
5. Distribución de features por vocal
6. Detección de outliers
7. Análisis de correlación
8. Separabilidad de clases (PCA)
9. Resumen y conclusiones del EDA

## 0. Importaciones

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy import stats

warnings.filterwarnings('ignore')

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

PROJECT_ROOT  = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR   = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Paleta de colores canónica para las 3 clases
CLASS_PALETTE = {
    'ELA_bulbar':    '#E05C4B',
    'ELA_no_bulbar': '#F4A236',
    'Control':       '#4B9CD3',
}

print(f'Figuras se guardarán en: {FIGURES_DIR}')

## 1. Carga y visión general del dataset

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'dataset_final.csv')

# Separar metadatos y features
META_COLS    = ['subject_id', 'genero', 'label_clinico', 'label_maquina']
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]
VOCALS       = ['a', 'e', 'i', 'o', 'u']

print('=== VISIÓN GENERAL ===')
print(f'Sujetos        : {len(df)}')
print(f'Features totales: {len(FEATURE_COLS)}  (5 vocales × 40 biomarcadores)')
print(f'Columnas totales: {df.shape[1]}')
print(f'\nPrimeras columnas: {list(df.columns[:8])}')
df.head()

In [ ]:
# Tipos de datos
print('=== TIPOS DE DATOS ===')
print(f'Metadatos   : {df[META_COLS].dtypes.to_dict()}')
print(f'Features    : {df[FEATURE_COLS].dtypes.value_counts().to_dict()}')

## 2. Distribución de clases (variable objetivo)

Analizar el desbalanceo es crítico antes del modelado. Si hay desbalanceo
significativo, justifica el uso de `class_weight='balanced'` o SMOTE en el ML.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, label_col, title in zip(
    axes,
    ['label_clinico', 'label_maquina'],
    ['Etiquetado clínico', 'Reetiquetado máquina (S4VM)'],
):
    counts = df[label_col].value_counts()
    bars   = ax.bar(
        counts.index,
        counts.values,
        color=[CLASS_PALETTE[c] for c in counts.index],
        edgecolor='white', linewidth=1.2,
    )
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel('N sujetos')
    ax.set_ylim(0, counts.max() * 1.25)

    for bar, count in zip(bars, counts.values):
        pct = count / len(df) * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{count}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=10,
        )

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_01_class_distribution.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_01_class_distribution.png')

In [ ]:
# Tabla resumen de distribución
print('=== DISTRIBUCIÓN DETALLADA ===')
for label_col in ['label_clinico', 'label_maquina']:
    print(f'\n{label_col}:')
    counts = df[label_col].value_counts()
    for cls, n in counts.items():
        ratio = n / len(df) * 100
        print(f'  {cls:20s}: {n:3d} sujetos ({ratio:.1f}%)')

# Sujetos reetiquetados
diff = df['label_clinico'] != df['label_maquina']
print(f'\nSujetos reetiquetados (clínico ≠ máquina): {diff.sum()}')
if diff.any():
    print(df.loc[diff, ['subject_id', 'label_clinico', 'label_maquina']].to_string(index=False))

## 3. Análisis demográfico

In [ ]:
# Distribución género × clase clínica
df['genero_str'] = df['genero'].map({0: 'Mujer', 1: 'Hombre'})

cross = pd.crosstab(df['label_clinico'], df['genero_str'])
print('=== GÉNERO × CLASE CLÍNICA ===')
print(cross)

fig, ax = plt.subplots(figsize=(8, 5))
cross.plot(
    kind='bar', ax=ax,
    color=['#E8A0A0', '#6B9EC7'],
    edgecolor='white', linewidth=1,
)
ax.set_title('Distribución de género por clase', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('N sujetos')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
ax.legend(title='Género')

# Añadir valores encima de cada barra
for container in ax.containers:
    ax.bar_label(container, padding=2, fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_02_gender_distribution.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_02_gender_distribution.png')

## 4. Calidad de datos

In [ ]:
print('=== CALIDAD DE DATOS ===')

# Nulos en features
null_counts = df[FEATURE_COLS].isnull().sum()
null_cols   = null_counts[null_counts > 0]
print(f'Valores nulos totales en features: {null_counts.sum()}')
if not null_cols.empty:
    print(f'Columnas con nulos:\n{null_cols}')
else:
    print('✓ Sin valores nulos en features acústicas.')

# Duplicados
n_dup = df.duplicated().sum()
print(f'\nFilas duplicadas: {n_dup}')
if n_dup == 0:
    print('✓ Sin duplicados.')

# Features con varianza cero (columnas constantes — inútiles para el modelo)
feat_df   = df[FEATURE_COLS].select_dtypes(include='number')
zero_var  = feat_df.columns[feat_df.var() == 0].tolist()
print(f'\nFeatures con varianza cero: {len(zero_var)}')
if zero_var:
    print(f'  → {zero_var}')
    print('  ⚠ Estas features se eliminarán antes del modelado.')
else:
    print('✓ Todas las features tienen varianza > 0.')

## 5. Distribución de features por vocal

Visualizamos la distribución de cada tipo de biomarcador en las 5 vocales.
Esto permite identificar qué vocales son más discriminativas entre clases.

In [ ]:
# Estadísticas descriptivas por vocal
print('=== ESTADÍSTICAS DESCRIPTIVAS POR VOCAL ===')
for vocal in VOCALS:
    cols_v = [c for c in FEATURE_COLS if c.endswith(f'_{vocal}')]
    stats_v = df[cols_v].describe()
    print(f'\nVocal "{vocal}" ({len(cols_v)} features):')
    print(stats_v.loc[['mean', 'std', 'min', 'max']].round(4).to_string())

In [ ]:
# Boxplots: feature H_tf (Entropía conjunta T-F) por vocal y clase
# Esta feature es uno de los biomarcadores más informativos del sistema
fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=False)

for ax, vocal in zip(axes, VOCALS):
    col = f'H_tf_{vocal}'
    if col not in df.columns:
        ax.set_visible(False)
        continue

    data_plot = df[['label_clinico', col]].copy()
    order     = ['ELA_bulbar', 'ELA_no_bulbar', 'Control']
    order     = [o for o in order if o in data_plot['label_clinico'].unique()]

    sns.boxplot(
        data=data_plot, x='label_clinico', y=col, order=order,
        palette=CLASS_PALETTE, ax=ax,
        linewidth=1, flierprops=dict(marker='o', markersize=3, alpha=0.5),
    )
    ax.set_title(f'Vocal "{vocal}"', fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels([o.replace('ELA_', '') for o in order], rotation=15, fontsize=8)
    if ax != axes[0]:
        ax.set_ylabel('')

fig.suptitle('Entropía conjunta T-F (H_tf) por vocal y clase', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_03_htf_by_vocal_class.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_03_htf_by_vocal_class.png')

In [ ]:
# Boxplots: energía de banda 1 (Enr_Bn1) por vocal y clase
fig, axes = plt.subplots(1, 5, figsize=(18, 5), sharey=False)

for ax, vocal in zip(axes, VOCALS):
    col = f'Enr_Bn1_{vocal}'
    if col not in df.columns:
        ax.set_visible(False)
        continue

    order = ['ELA_bulbar', 'ELA_no_bulbar', 'Control']
    order = [o for o in order if o in df['label_clinico'].unique()]

    sns.boxplot(
        data=df, x='label_clinico', y=col, order=order,
        palette=CLASS_PALETTE, ax=ax, linewidth=1,
        flierprops=dict(marker='o', markersize=3, alpha=0.5),
    )
    ax.set_title(f'Vocal "{vocal}"', fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels([o.replace('ELA_', '') for o in order], rotation=15, fontsize=8)
    if ax != axes[0]:
        ax.set_ylabel('')

fig.suptitle('Energía de banda 1 (Enr_Bn1) por vocal y clase', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_04_enrbn1_by_vocal_class.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_04_enrbn1_by_vocal_class.png')

## 6. Detección de outliers

Se usa el método IQR (rango intercuartílico). Un valor es outlier si está
más allá de Q1 − 1.5·IQR o Q3 + 1.5·IQR.

In [ ]:
def count_outliers_iqr(series: pd.Series) -> int:
    """Cuenta outliers por método IQR."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())

feat_num = df[FEATURE_COLS].select_dtypes(include='number')
outlier_counts = feat_num.apply(count_outliers_iqr)
top_outliers   = outlier_counts.nlargest(20)

print('=== FEATURES CON MÁS OUTLIERS (top 20) ===')
print(top_outliers.to_string())
print(f'\nTotal outliers (acumulado): {outlier_counts.sum()}')
print(f'Features con ≥5 outliers: {(outlier_counts >= 5).sum()}')

In [ ]:
# Visualizar outliers en las 20 features más problemáticas
top_feat = top_outliers.index.tolist()

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(
    range(len(top_feat)), top_outliers.values,
    color='#5B8DB8', edgecolor='white',
)
ax.set_yticks(range(len(top_feat)))
ax.set_yticklabels(top_feat, fontsize=9)
ax.set_xlabel('Número de outliers (IQR)')
ax.set_title('Top 20 features por outliers', fontweight='bold')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_05_outliers.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_05_outliers.png')

## 7. Análisis de correlación

Alta correlación entre features puede indicar redundancia. Features con
correlación > 0.95 son candidatas a eliminación antes del modelado.

In [ ]:
# Correlación dentro de cada vocal (reducido para visualización clara)
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, vocal in zip(axes, VOCALS):
    cols_v = [c for c in FEATURE_COLS if c.endswith(f'_{vocal}')]
    corr_v = df[cols_v].corr()

    # Etiquetas cortas (sin sufijo vocal)
    short_labels = [c.replace(f'_{vocal}', '') for c in cols_v]

    mask = np.triu(np.ones_like(corr_v, dtype=bool))
    sns.heatmap(
        corr_v, ax=ax, mask=mask,
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        xticklabels=short_labels, yticklabels=short_labels,
        linewidths=0, cbar=ax == axes[-1],
        annot=False,
    )
    ax.set_title(f'Vocal "{vocal}"', fontweight='bold', fontsize=10)
    ax.tick_params(axis='both', labelsize=5)

fig.suptitle('Correlación de Pearson entre features por vocal', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_06_correlation_by_vocal.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_06_correlation_by_vocal.png')

In [ ]:
# Pares altamente correlacionados (|r| > 0.95) — candidatos a eliminar
corr_matrix = feat_num.corr().abs()
upper_tri   = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))
high_corr   = upper_tri.stack()
high_corr   = high_corr[high_corr > 0.95].reset_index()
high_corr.columns = ['feature_1', 'feature_2', 'correlacion']
high_corr = high_corr.sort_values('correlacion', ascending=False)

print(f'=== PARES CON |r| > 0.95: {len(high_corr)} ===')
if not high_corr.empty:
    print(high_corr.to_string(index=False))
    print(f'\n→ {high_corr["feature_2"].nunique()} features candidatas a eliminar por redundancia.')
else:
    print('No hay pares con correlación > 0.95.')

## 8. Separabilidad de clases — PCA

La reducción de dimensionalidad mediante PCA permite visualizar si las clases
son linealmente separables en el espacio de features acústicas.
Si las clases se solapan mucho en 2D, los modelos lineales tendrán dificultades;
esto justifica el uso de modelos de ensemble no lineales (Random Forest, XGBoost).

In [ ]:
# Preparar datos para PCA — imputar nulos si los hay
X = feat_num.fillna(feat_num.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA a 2 y 3 componentes
pca2 = PCA(n_components=2, random_state=42)
pca3 = PCA(n_components=3, random_state=42)
X_2d = pca2.fit_transform(X_scaled)
X_3d = pca3.fit_transform(X_scaled)

var_2d = pca2.explained_variance_ratio_.sum() * 100
var_3d = pca3.explained_variance_ratio_.sum() * 100
print(f'PCA 2 componentes: {var_2d:.1f}% varianza explicada')
print(f'PCA 3 componentes: {var_3d:.1f}% varianza explicada')

In [ ]:
# Scatter PCA 2D — etiquetado clínico vs reetiquetado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, label_col, title in zip(
    axes,
    ['label_clinico', 'label_maquina'],
    ['Etiquetado clínico', 'Reetiquetado máquina (S4VM)'],
):
    for cls in ['ELA_bulbar', 'ELA_no_bulbar', 'Control']:
        mask = df[label_col] == cls
        if not mask.any():
            continue
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=CLASS_PALETTE[cls], label=cls,
            alpha=0.75, edgecolors='white', linewidth=0.5, s=60,
        )

    ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(title, fontweight='bold')
    ax.legend(title='Clase', fontsize=9)
    ax.set_aspect('auto')

fig.suptitle(
    f'PCA 2D — {var_2d:.1f}% varianza explicada\n'
    f'(dataset acústico completo, 5 vocales)',
    fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_07_pca2d.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_07_pca2d.png')

In [ ]:
# Varianza explicada acumulada — cuántos componentes necesitamos
pca_full = PCA(random_state=42).fit(X_scaled)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_) * 100

n_80 = np.argmax(cumvar >= 80) + 1
n_90 = np.argmax(cumvar >= 90) + 1
n_95 = np.argmax(cumvar >= 95) + 1

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cumvar) + 1), cumvar, color='#4B9CD3', linewidth=2)
ax.fill_between(range(1, len(cumvar) + 1), cumvar, alpha=0.15, color='#4B9CD3')

for n_comp, pct, color in [(n_80, 80, '#E05C4B'), (n_90, 90, '#F4A236'), (n_95, 95, '#5A8F59')]:
    ax.axhline(y=pct, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(x=n_comp, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.text(n_comp + 1, pct - 3, f'{pct}% → {n_comp} comp.', fontsize=9, color=color)

ax.set_xlabel('Número de componentes principales')
ax.set_ylabel('Varianza explicada acumulada (%)')
ax.set_title('Varianza explicada acumulada — PCA', fontweight='bold')
ax.set_xlim(1, min(80, len(cumvar)))
ax.set_ylim(0, 102)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_08_pca_variance.png', bbox_inches='tight')
plt.show()

print(f'Componentes para 80% varianza: {n_80}')
print(f'Componentes para 90% varianza: {n_90}')
print(f'Componentes para 95% varianza: {n_95}')
print('✓ Figura guardada: eda_08_pca_variance.png')

## 9. Test de significancia estadística

Test de Kruskal-Wallis (no paramétrico) para identificar qué features
diferencian significativamente las clases. Complementa el análisis visual.

In [ ]:
from scipy.stats import kruskal

# Calcular p-valor de Kruskal-Wallis para cada feature × label_clinico
classes   = df['label_clinico'].unique()
kw_results = []

for feat in FEATURE_COLS:
    groups = [df.loc[df['label_clinico'] == cls, feat].dropna() for cls in classes]
    if any(len(g) < 2 for g in groups):
        continue
    try:
        stat, pval = kruskal(*groups)
        kw_results.append({'feature': feat, 'H_stat': stat, 'p_value': pval})
    except Exception:
        pass

df_kw = pd.DataFrame(kw_results).sort_values('p_value')

# Corrección de Bonferroni
alpha         = 0.05
alpha_bonf    = alpha / len(df_kw)
n_significant = (df_kw['p_value'] < alpha_bonf).sum()

print(f'=== TEST KRUSKAL-WALLIS (etiquetado clínico) ===')
print(f'Alpha Bonferroni: {alpha_bonf:.6f}')
print(f'Features significativas: {n_significant} / {len(df_kw)}')
print('\nTop 20 features más discriminativas:')
print(df_kw.head(20).to_string(index=False))

In [ ]:
# Visualizar top 25 features más significativas
top25 = df_kw.head(25).copy()
top25['-log10(p)'] = -np.log10(top25['p_value'].clip(lower=1e-300))

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E05C4B' if p < alpha_bonf else '#A0B8CF' for p in top25['p_value']]
ax.barh(range(len(top25)), top25['-log10(p)'], color=colors, edgecolor='white')
ax.axvline(
    x=-np.log10(alpha_bonf),
    color='#333333', linestyle='--', linewidth=1.2,
    label=f'Umbral Bonferroni (α={alpha_bonf:.4f})',
)
ax.set_yticks(range(len(top25)))
ax.set_yticklabels(top25['feature'], fontsize=8)
ax.set_xlabel('-log₁₀(p-valor)')
ax.set_title('Top 25 features discriminativas (Kruskal-Wallis)', fontweight='bold')
ax.legend(fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_09_kruskal_wallis.png', bbox_inches='tight')
plt.show()
print('✓ Figura guardada: eda_09_kruskal_wallis.png')

## 10. Resumen y conclusiones del EDA

In [ ]:
print('=' * 65)
print('RESUMEN DEL ANÁLISIS EXPLORATORIO DE DATOS')
print('=' * 65)

print(f'\n[Dataset]')
print(f'  Sujetos totales  : {len(df)}')
print(f'  Features acústicas: {len(FEATURE_COLS)}')

print(f'\n[Distribución de clases — clínico]')
for cls, n in df['label_clinico'].value_counts().items():
    print(f'  {cls:20s}: {n} ({n/len(df)*100:.1f}%)')

print(f'\n[Distribución de clases — máquina]')
for cls, n in df['label_maquina'].value_counts().items():
    print(f'  {cls:20s}: {n} ({n/len(df)*100:.1f}%)')

diff_count = (df['label_clinico'] != df['label_maquina']).sum()
print(f'  Sujetos reetiquetados: {diff_count}')

print(f'\n[Calidad de datos]')
print(f'  Nulos en features: {df[FEATURE_COLS].isnull().sum().sum()}')
print(f'  Duplicados       : {df.duplicated().sum()}')
print(f'  Varianza cero    : {len(zero_var)} features')

print(f'\n[Outliers]')
print(f'  Features con ≥5 outliers (IQR): {(outlier_counts >= 5).sum()}')

print(f'\n[Correlación]')
print(f'  Pares con |r| > 0.95: {len(high_corr)}')

print(f'\n[PCA]')
print(f'  Varianza en 2 componentes: {var_2d:.1f}%')
print(f'  Componentes para 95% var : {n_95}')

print(f'\n[Significancia estadística]')
print(f'  Features significativas (Bonferroni): {n_significant}')

print(f'\n[Figuras generadas]')
for fig_file in sorted(FIGURES_DIR.glob('eda_*.png')):
    print(f'  {fig_file.name}')

print('\n' + '=' * 65)
print('→ Siguiente paso: notebooks/ml_experiments.ipynb')
print('=' * 65)

---
## Checklist para la memoria (Capítulo 3 y 5)

Los resultados de este EDA documentan y justifican:

| Hallazgo | Implicación metodológica |
|----------|-------------------------|
| Desbalanceo de clases | Usar `class_weight='balanced'` en todos los clasificadores |
| Outliers detectados | Aplicar StandardScaler (robusto) o RobustScaler antes del modelado |
| Pares con \|r\| > 0.95 | Considerar eliminación de features redundantes |
| PCA → varianza en pocos componentes | Reducción de dimensionalidad como preprocessing opcional |
| Features Kruskal significativas | Justifica el uso de feature importance en Random Forest y XGBoost |
| Solapamiento visual en PCA 2D | Justifica modelos no lineales sobre modelos lineales |

**Siguiente paso:** `notebooks/ml_experiments.ipynb`